# 😴 Detección de somnolencia en conductores con visión por computadora

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndresNenger/deteccion-somnolencia-ear/blob/main/notebooks/01_deteccion_somnolencia_colab.ipynb)

**Flujo del proyecto**

`Video de entrada → Detección del rostro → Localización de los ojos → Cálculo del EAR → Detección de somnolencia → Video de salida con alerta`

Usaremos **MediaPipe Face Landmarker** (478 puntos faciales) y el **Eye Aspect Ratio (EAR)**:

$$EAR = \frac{\lVert p_2 - p_6 \rVert + \lVert p_3 - p_5 \rVert}{2\,\lVert p_1 - p_4 \rVert}$$

- EAR ≥ 0.22 → ojos abiertos (conductor despierto)
- EAR < 0.22 durante más de 15 fotogramas consecutivos → **SOMNOLIENTO** (alerta)

> ⚠️ Proyecto educativo. **No es un dispositivo de seguridad vehicular** ni un producto certificado.

## Paso 0 · Instalar dependencias

In [ ]:
!pip install -q mediapipe

## Paso 1 · Video de entrada

Elige de dónde viene el video con la variable `FUENTE`:

| Valor | Qué hace |
|---|---|
| `"subir"` | Abre un selector para subir un archivo desde tu computador |
| `"drive"` | Monta Google Drive y usa la ruta `RUTA_VIDEO` |
| `"url"` | Descarga el video desde un enlace directo (`URL_VIDEO`) |
| `"ruta"` | Usa un archivo ya presente, p. ej. `/content/videoplayback.mp4` |

Recomendación: un video de **cara frontal**, buena iluminación, 10–60 s. Usa solo videos propios o con permiso de las personas que aparecen.

In [ ]:
FUENTE = "subir"                       # "subir" | "drive" | "url" | "ruta"
RUTA_VIDEO = "/content/videoplayback.mp4"   # para "drive" o "ruta"
URL_VIDEO = ""                         # para "url"

import os

if FUENTE == "subir":
    from google.colab import files
    subido = files.upload()
    RUTA_VIDEO = "/content/" + next(iter(subido))
elif FUENTE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")      # RUTA_VIDEO p. ej. /content/drive/MyDrive/video.mp4
elif FUENTE == "url":
    import urllib.request
    RUTA_VIDEO = "/content/video_entrada.mp4"
    urllib.request.urlretrieve(URL_VIDEO, RUTA_VIDEO)

print("Video:", RUTA_VIDEO, "-", round(os.path.getsize(RUTA_VIDEO) / 1e6, 1), "MB")

### Comprobar que el video se abre y mostrar el primer fotograma

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(RUTA_VIDEO)
if not cap.isOpened():
    print("Error: No se pudo abrir el video")
else:
    print("Video abierto correctamente")

FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
ANCHO = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ALTO = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"{ANCHO}x{ALTO} px · {FPS:.1f} fps · {TOTAL} fotogramas · {TOTAL / FPS:.1f} s")

ret, frame0 = cap.read()
cap.release()
plt.imshow(cv2.cvtColor(frame0, cv2.COLOR_BGR2RGB))
plt.title("Primer fotograma")
plt.axis("off")
plt.show()

## Paso 2 · Lectura de cada fotograma

OpenCV entrega los fotogramas en **BGR**; MediaPipe espera **RGB**. El bucle principal (Paso 7) hace la conversión en cada fotograma:

```python
ret, frame = cap.read()
if not ret:
    break
rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
```

## Paso 3 · Detección del rostro (MediaPipe Face Landmarker)

Descargamos el modelo `face_landmarker.task` y creamos el detector en **modo VIDEO** (usa seguimiento entre fotogramas y exige marcas de tiempo crecientes).

In [ ]:
import urllib.request
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

URL_MODELO = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
              "face_landmarker/float16/1/face_landmarker.task")
RUTA_MODELO = "face_landmarker.task"
if not os.path.exists(RUTA_MODELO):
    urllib.request.urlretrieve(URL_MODELO, RUTA_MODELO)


def crear_detector():
    opciones = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=RUTA_MODELO),
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1,
    )
    return vision.FaceLandmarker.create_from_options(opciones)


def obtener_landmarks(detector, frame_bgr, t_ms):
    """Devuelve array (478, 2) normalizado 0-1, o None si no hay rostro."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    resultado = detector.detect_for_video(imagen, int(t_ms))
    if not resultado.face_landmarks:
        return None
    return np.array([[p.x, p.y] for p in resultado.face_landmarks[0]])


with crear_detector() as det:
    lm0 = obtener_landmarks(det, frame0, 0)

if lm0 is None:
    print("No se detectó ningún rostro en el primer fotograma")
else:
    print(f"Rostro detectado: {len(lm0)} puntos de referencia")

## Paso 4 · Localización de los ojos

Cada ojo se describe con **6 puntos** de la malla (en el orden p1…p6 de la fórmula del EAR):

| Ojo | Índices |
|---|---|
| Izquierdo | `[33, 160, 158, 133, 153, 144]` |
| Derecho | `[362, 385, 387, 263, 373, 380]` |

> Los landmarks vienen normalizados (0–1). Los pasamos a **píxeles** antes de medir distancias; si no, la relación de aspecto del video deforma el EAR.

In [ ]:
OJO_IZQ = [33, 160, 158, 133, 153, 144]
OJO_DER = [362, 385, 387, 263, 373, 380]


def a_pixeles(lm, ancho, alto):
    return lm * np.array([ancho, alto])


if lm0 is not None:
    px0 = a_pixeles(lm0, ANCHO, ALTO)
    vista = frame0.copy()
    for idx in OJO_IZQ + OJO_DER:
        x, y = px0[idx]
        cv2.circle(vista, (int(x), int(y)), 3, (0, 255, 0), -1)
    plt.figure(figsize=(8, 5))
    plt.imshow(cv2.cvtColor(vista, cv2.COLOR_BGR2RGB))
    plt.title("Los 6 puntos de cada ojo")
    plt.axis("off")
    plt.show()

## Paso 5 · Cálculo del EAR

- `A = ‖p2 − p6‖` (distancia vertical 1-5)
- `B = ‖p3 − p5‖` (distancia vertical 2-4)
- `C = ‖p1 − p4‖` (distancia horizontal 0-3)

$EAR = (A + B) / (2C)$, y se promedian los dos ojos.

In [ ]:
def ear_ojo(p):
    A = np.linalg.norm(p[1] - p[5])
    B = np.linalg.norm(p[2] - p[4])
    C = np.linalg.norm(p[0] - p[3])
    return (A + B) / (2.0 * C) if C > 0 else 0.0


def ear_promedio(px):
    return (ear_ojo(px[OJO_IZQ]) + ear_ojo(px[OJO_DER])) / 2.0


if lm0 is not None:
    print(f"EAR del primer fotograma: {ear_promedio(px0):.3f}")

## Paso 6 · Detección de somnolencia (umbral + contador)

| Parámetro | Valor | Significado |
|---|---|---|
| `UMBRAL_EAR` | 0.22 | Por debajo, el ojo se considera cerrado |
| `FRAMES_CONSECUTIVOS` | 15 | Cuántos fotogramas seguidos con ojo cerrado activan la alerta |

Si el EAR vuelve a subir, el contador se **reinicia** (un parpadeo normal dura ~3–8 fotogramas a 30 fps y no dispara la alerta).

> 15 fotogramas = 0,5 s a 30 fps, pero 1 s a 15 fps. Para fijar el tiempo en segundos usa `FRAMES_CONSECUTIVOS = round(SEGUNDOS * FPS)`.

In [ ]:
UMBRAL_EAR = 0.22
FRAMES_CONSECUTIVOS = 15
# Alternativa por tiempo (descomenta): FRAMES_CONSECUTIVOS = max(1, round(0.5 * FPS))

print(f"Alerta si EAR < {UMBRAL_EAR} por más de {FRAMES_CONSECUTIVOS} fotogramas "
      f"(≈ {FRAMES_CONSECUTIVOS / FPS:.2f} s a {FPS:.1f} fps)")

## Paso 7 · Bucle principal: alerta sobre cada fotograma

Se recorre el video completo, se calcula el EAR, se actualiza el contador y se dibuja la información. Los textos no llevan tildes porque `cv2.putText` no las soporta.

In [ ]:
SALIDA = "resultado.mp4"

cap = cv2.VideoCapture(RUTA_VIDEO)
writer = cv2.VideoWriter(SALIDA, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (ANCHO, ALTO))

contador = 0
estado = "DESPIERTO"
registros = []

with crear_detector() as detector:
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        lm = obtener_landmarks(detector, frame, idx * 1000 / FPS)
        ear = None
        if lm is not None:
            px = a_pixeles(lm, ANCHO, ALTO)
            ear = ear_promedio(px)
            for i in OJO_IZQ + OJO_DER:
                cv2.circle(frame, (int(px[i][0]), int(px[i][1])), 2, (0, 255, 0), -1)

        # --- lógica de somnolencia ---
        if ear is None:
            contador = 0
            estado = "SIN ROSTRO"
        else:
            contador = contador + 1 if ear < UMBRAL_EAR else 0
            estado = "SOMNOLIENTO" if contador > FRAMES_CONSECUTIVOS else "DESPIERTO"

        # --- dibujo ---
        cv2.putText(frame, "EAR: " + ("--" if ear is None else f"{ear:.2f}"), (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        cv2.putText(frame, f"Contador: {contador}/{FRAMES_CONSECUTIVOS}", (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
        cv2.putText(frame, estado, (20, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.9,
                    (0, 0, 255) if estado == "SOMNOLIENTO" else (0, 255, 0), 2)
        if estado == "SOMNOLIENTO":
            cv2.putText(frame, "ALERTA", (ANCHO // 2 - 110, ALTO // 2),
                        cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 255), 5)

        writer.write(frame)
        registros.append({"frame": idx, "tiempo_s": idx / FPS, "ear": ear,
                          "contador": contador, "estado": estado})
        idx += 1

cap.release()
writer.release()
print("Video terminado.", idx, "fotogramas procesados")

## Paso 8 · Video de salida

OpenCV guarda con el códec `mp4v`, que el navegador no reproduce. Lo reconvertimos a **H.264** con ffmpeg (ya instalado en Colab) y lo mostramos aquí mismo.

In [ ]:
import subprocess, base64, shutil
from IPython.display import HTML, display

if shutil.which("ffmpeg"):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", SALIDA, "-vcodec", "libx264",
                    "-pix_fmt", "yuv420p", "resultado_h264.mp4"], check=True)
    os.replace("resultado_h264.mp4", SALIDA)

tam_mb = os.path.getsize(SALIDA) / 1e6
if tam_mb < 25:
    datos = base64.b64encode(open(SALIDA, "rb").read()).decode()
    display(HTML(f'<video width="640" controls><source src="data:video/mp4;base64,{datos}" '
                 f'type="video/mp4"></video>'))
else:
    print(f"El video pesa {tam_mb:.0f} MB: descárgalo con la celda siguiente para verlo.")

## Análisis · EAR a lo largo del tiempo

In [ ]:
import pandas as pd

df = pd.DataFrame(registros)
con_rostro = df["ear"].notna().sum()
frames_alerta = (df["estado"] == "SOMNOLIENTO").sum()
eventos = ((df["estado"] == "SOMNOLIENTO") & (df["estado"].shift() != "SOMNOLIENTO")).sum()

print(f"Fotogramas totales:      {len(df)}")
print(f"Con rostro detectado:    {con_rostro} ({100 * con_rostro / max(len(df), 1):.1f} %)")
print(f"Fotogramas en alerta:    {frames_alerta}")
print(f"Eventos de alerta:       {eventos}")

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(df["tiempo_s"], df["ear"], lw=1, label="EAR")
ax.axhline(UMBRAL_EAR, color="red", ls="--", label=f"Umbral {UMBRAL_EAR}")
ax.fill_between(df["tiempo_s"], 0, 1, where=df["estado"] == "SOMNOLIENTO",
                color="red", alpha=0.2, transform=ax.get_xaxis_transform(), label="Alerta")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("EAR")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
df.to_csv("ear.csv", index=False)

try:
    from google.colab import files
    files.download(SALIDA)     # resultado.mp4
    files.download("ear.csv")
except ImportError:
    print("Fuera de Colab: los archivos quedaron en", os.getcwd())

## Calibración y límites

- **El umbral 0.22 es un punto de partida, no una verdad universal.** Depende de la persona (forma de los ojos), de la cámara y del ángulo. Mira la gráfica de arriba: el EAR con ojos abiertos debería quedar claramente por encima del umbral y los cierres claramente por debajo. Si no, ajusta `UMBRAL_EAR`.
- **Gafas, reflejos, poca luz o la cabeza girada** degradan los landmarks y pueden causar falsos positivos o negativos.
- El EAR solo mide **cierre ocular**. La somnolencia real también se manifiesta en bostezos, cabeceo y conducción errática.
- Esto es una **demostración educativa**: no sustituye sistemas homologados de monitoreo del conductor ni el descanso adecuado.

Siguiente paso: lee el tutorial completo en la carpeta `docs/` del repositorio.